# Prompt Registration
Register or update the system prompt in Vertex AI Prompt Management.
Run this notebook once to create the prompt, then copy the `prompt_id` into `gcp_rag.ipynb`.

In [1]:
# !gcloud auth application-default login

In [2]:
import google.auth
import vertexai
from vertexai import types, generative_models

_, project_id = google.auth.default()
LOCATION = "us-central1"

# Instantiate the Agent Platform client (new SDK)
client = vertexai.Client(project=project_id, location=LOCATION)
print(f"Project ID : {project_id}")
print(f"Client     : {client}")


Project ID : dev-mq-tech-transfer
Client     : <vertexai._genai.client.Client object at 0x0000026573D7BD40>


In [3]:
PROMPT_DISPLAY_NAME = "hello-world-prompt"
MODEL_NAME          = "gemini-2.5-flash"

# Content/Part live in google.genai.types, not vertexai.types
from google.genai import types as genai_types

# Version 1: simple hello world
_SYSTEM_INSTRUCTION_V1 = (
    "You are a friendly assistant. "
    "When greeted, respond with 'Hello, World!' and briefly introduce yourself."
)

# Version 2: enthusiastic hello world
_SYSTEM_INSTRUCTION_V2 = (
    "You are an enthusiastic and cheerful assistant. "
    "When greeted, respond with 'Hello, World!' using excitement and emojis, "
    "then warmly invite the user to ask you anything."
)

def make_prompt(system_instruction):
    return types.Prompt(
        prompt_data=types.PromptData(
            model=MODEL_NAME,
            system_instruction=genai_types.Content(
                role="user",
                parts=[genai_types.Part(text=system_instruction)],
            ),
            contents=[
                genai_types.Content(
                    role="user",
                    parts=[genai_types.Part(text="Hello!")],
                )
            ],
        ),
    )

print("Prompt templates defined (V1 and V2).")


Prompt templates defined (V1 and V2).


In [4]:
from google.genai import types as genai_types

KMS_KEY_NAME = (
    "projects/dev-mq-tech-transfer/locations/us-central1"
    "/keyRings/poc/cryptoKeys/poc"
)

_enc_spec = genai_types.EncryptionSpec(kms_key_name=KMS_KEY_NAME)
_create_cfg = types.CreatePromptVersionConfig(encryption_spec=_enc_spec)

# Register Version 1: create the prompt resource (with first version)
saved_v1 = client.prompts.create_version(
    prompt=make_prompt(_SYSTEM_INSTRUCTION_V1),
    config=_create_cfg,
)
REGISTERED_PROMPT_ID = saved_v1.prompt_id
print(f"Version 1 created  →  prompt_id: {REGISTERED_PROMPT_ID}  |  version_id: {saved_v1.version_id}")

# Register Version 2: add a second version to the same prompt resource via update()
saved_v2 = client.prompts.update(
    prompt_id=REGISTERED_PROMPT_ID,
    prompt=make_prompt(_SYSTEM_INSTRUCTION_V2),
)
print(f"Version 2 added    →  prompt_id: {saved_v2.prompt_id}  |  version_id: {saved_v2.version_id}")

print(f"\nView at: https://console.cloud.google.com/vertex-ai/generative/language/prompts?project={project_id}")


Version 1 created  →  prompt_id: 527730671838298112  |  version_id: 1


Version 2 added    →  prompt_id: 527730671838298112  |  version_id: 2

View at: https://console.cloud.google.com/vertex-ai/generative/language/prompts?project=dev-mq-tech-transfer


## Get a Saved Prompt

Retrieves the latest version of a prompt by its ID.

In [5]:
# Get the latest version of the prompt
retrieved_prompt = client.prompts.get(prompt_id=REGISTERED_PROMPT_ID)

print(f"prompt_id  : {retrieved_prompt.prompt_id}")
print(f"model      : {retrieved_prompt.prompt_data.model}")
print(f"system_instruction ({len(str(retrieved_prompt.prompt_data.system_instruction))} chars)")


prompt_id  : 527730671838298112
model      : gemini-2.5-flash
system_instruction (205 chars)


## Get a Specific Prompt Version

Use `get_version` when you need to pin to a specific version rather than always using the latest.

In [6]:
# List all versions and print version_id + system instruction for each
versions_meta = list(client.prompts.list_versions(prompt_id=REGISTERED_PROMPT_ID))
print(f"Total versions registered: {len(versions_meta)}\n")

for v in versions_meta:
    vp = client.prompts.get_version(
        prompt_id=REGISTERED_PROMPT_ID,
        version_id=v.version_id,
    )
    sys_instr = vp.prompt_data.system_instruction
    # system_instruction is a genai_types.Content object with .parts list
    if sys_instr is None:
        text = "(none)"
    elif hasattr(sys_instr, "parts") and sys_instr.parts:
        text = " ".join(
            p.text for p in sys_instr.parts if hasattr(p, "text") and p.text
        )
    else:
        text = str(sys_instr)

    print(f"version_id : {v.version_id}")
    print(f"prompt     : {text}")
    print()


Total versions registered: 2



version_id : 1
prompt     : You are an enthusiastic and cheerful assistant. When greeted, respond with 'Hello, World!' using excitement and emojis, then warmly invite the user to ask you anything.



version_id : 2
prompt     : You are an enthusiastic and cheerful assistant. When greeted, respond with 'Hello, World!' using excitement and emojis, then warmly invite the user to ask you anything.



## List All Prompts in the Project

Lists every prompt resource saved under the current GCP project.

In [7]:
# List all prompt resources in the project
prompt_refs = list(client.prompts.list())
print(f"Total prompts in project: {len(prompt_refs)}\n")
for ref in prompt_refs:
    p = client.prompts.get(prompt_id=ref.prompt_id)
    print(f"  prompt_id : {ref.prompt_id}")
    print(f"  model     : {p.prompt_data.model}")
    print()


Total prompts in project: 12



  prompt_id : 527730671838298112
  model     : gemini-2.5-flash



  prompt_id : 6035633016112414720
  model     : gemini-2.5-flash



  prompt_id : 2368014069572567040
  model     : gemini-2.5-flash

  prompt_id : 4982353653261139968
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



  prompt_id : 1651941728820658176
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



...

  prompt_id : 2258379566143766528
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



  prompt_id : 34727250129584128
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



  prompt_id : 6646011503109472256
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



  prompt_id : 454687915381882880
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash

  prompt_id : 7336188146004000768
  model     : projects/dev-mq-tech-transfer/locations/us-central1/publishers/google/models/gemini-2.5-flash



  prompt_id : 6136182529838809088
  model     : projects/dev-mq-tech-transfer/locations/global/publishers/google/models/gemini-2.5-flash-lite



## Restore a Prompt Version

Promotes an older version back to the latest. Useful for rolling back a bad prompt update.

In [8]:
# Restore a previous version as the latest
# Set RESTORE_VERSION_ID to the version you want to promote
RESTORE_VERSION_ID = "1"   # change to your target version

restored_prompt = client.prompts.restore_version(
    prompt_id=REGISTERED_PROMPT_ID,
    version_id=RESTORE_VERSION_ID,
)
print(f"Version {RESTORE_VERSION_ID} restored as latest for prompt {restored_prompt.prompt_id}")


Version 1 restored as latest for prompt 527730671838298112


## Delete a Version / Delete a Prompt

> ⚠️ `delete` removes the entire prompt resource and **all** its versions permanently. `delete_version` removes only one version. Both are commented out below — uncomment carefully.

In [9]:
DELETE_VERSION_ID = "1"   # version to remove

# Delete a single version (keeps the prompt resource and other versions)
# client.prompts.delete_version(
#     prompt_id=REGISTERED_PROMPT_ID,
#     version_id=DELETE_VERSION_ID,
# )
# print(f"Version {DELETE_VERSION_ID} deleted from prompt {REGISTERED_PROMPT_ID}")

# Delete the entire prompt resource and ALL its versions — irreversible
# client.prompts.delete(prompt_id=REGISTERED_PROMPT_ID)
# print(f"Prompt {REGISTERED_PROMPT_ID} and all versions deleted")


## Hello World — verify the prompt loads and the model responds

In [10]:
# Load the saved prompt back from Vertex AI
retrieved = client.prompts.get(prompt_id=REGISTERED_PROMPT_ID)
print(f"Loaded prompt  : {retrieved.prompt_id}")
print(f"Model          : {retrieved.prompt_data.model}")

# Extract the system instruction text from the Content object
sys_instr = retrieved.prompt_data.system_instruction
if sys_instr and hasattr(sys_instr, "parts") and sys_instr.parts:
    sys_text = " ".join(p.text for p in sys_instr.parts if hasattr(p, "text") and p.text)
else:
    sys_text = str(sys_instr) if sys_instr else None

# Run a simple hello-world generation using the stored system instruction text
model = generative_models.GenerativeModel(
    model_name=retrieved.prompt_data.model,
    system_instruction=sys_text,
)

response = model.generate_content("Hello! Who are you and what can you help me with?")
print("\n--- Model response ---")
print(response.text)


Loaded prompt  : 527730671838298112
Model          : gemini-2.5-flash


C:\Users\L137844\Downloads\gap_assesment\tredence_gap_assessment\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()



--- Model response ---
Hello, World!

I am a friendly AI assistant, and I'm here to help you with a wide range of tasks. I can answer questions, provide information, brainstorm ideas, help with writing, summarize texts, and much more. Just let me know what you need!


---
## Register a Custom Prompt (standalone)
Fill in the three variables below and run this cell on its own — no other cells need to be run first.

In [ ]:

# ── Standalone: register any prompt ───────────────────────────────────────────
# Set MY_PROMPT_NAME and MY_PROMPT_TEXT, then run.
# - If a prompt with that display name already exists → adds a new version.
# - If no prompt with that name is found       → creates a new prompt resource (version 1).

MY_PROMPT_NAME = "facilities_generated"   # display name shown in Vertex AI console

MY_PROMPT_TEXT ="""\
You are a pharmaceutical technology-transfer analyst. Your job is to reconstruct the
FACILITY section of a Technology-Transfer Gap Assessment by retrieving from and reasoning
over a document datastore, then emitting a single structured JSON object.

## What a gap assessment captures
For the evaluated area, compare four things:
- sending_site_requirements: product/process characteristics inherited from the sending
  process that the receiving facility must satisfy.
- global_requirements: corporate, pharmacopeial, and regulatory standards the facility must
  meet (e.g., EU GMP Annex 1, internal global quality standards, USP chapters, internal guidance).
- receiving_site_evidence: what the receiving site has designed, installed, qualified, or
  otherwise demonstrated.
- gaps: requirements not yet satisfied by evidence — missing, pending, or not-yet-available —
  and risks introduced by the receiving context that require controls not yet verified.

## Retrieval strategy (this corpus is deliberately hard to retrieve from)
Do NOT rely on one broad query. Decompose the Facility section into atomic facts, issue a
separate targeted retrieval query for each, then synthesize.

1. One fact per query. Cover at minimum: design standards & capacity; room classification
   (grades); air-pressurization / room pressure differentials (query EACH boundary separately —
   isolator↔surrounding room, air-lock↔room, grade↔grade, room↔corridor, containment enclosure);
   personnel/material/waste/equipment flows; raw-material & product handling and dispensing;
   environmental temperature (query EACH context separately: formulation WFI, general room
   ambient, cold storage/freezer, EM incubation); environmental humidity (general-room band and
   the dispensing limit are SEPARATE); cross-contamination / product mix-up; shared-building /
   multi-product context; utilities (WFI, clean steam, process air); cleaning & sanitising agents;
   isolator bio-decontamination (the decontaminant WORKING CONCENTRATION and the RESIDUAL
   PRODUCT-EXPOSURE LIMIT are different quantities — query each); cold storage / freezer; overall
   qualification status and milestones.

2. Rotate vocabulary. The same concept is worded differently across documents; if a query
   returns little, re-issue with synonyms:
     humidity ↔ "relative humidity" / "moisture level" / "HR"
     cross-contamination ↔ "product carryover" / "cross contamination"
     pressure ↔ "air-pressurization scheme" / "inter-room pressure regime" / "room pressure differentials"
     hydrogen peroxide ↔ "VPHP" / "vapour-phase hydrogen peroxide" / "vaporized hydrogen peroxide"
   A value may be STATED in one document and only REFERRED TO (without the number) in another.
   Trust the document that states the actual value; a pointer without a value is not the answer.

3. Name the context for any quantity that has several context-specific values (temperature,
   humidity, differential pressure). A bare "what is the temperature/humidity/pressure" is
   ambiguous and MUST be split by context or boundary.

4. Reassemble split tables. A parameter table may be chunked so column headers and data rows
   are retrieved separately (one chunk has labels but no numbers; another has numbers but no
   labels). Retrieve both and rejoin by position.

5. Follow multi-hop chains. A single topic may span documents: a hazard stated in one, the
   control in another, the verification in a third. Retrieve all links before concluding
   (e.g., cross-contamination: co-location hazard → segregated flows + pressure cascade control
   → airflow-visualisation / EM / pressure-differential verification).

6. Run negative-control queries. For anything that seems like it should exist, confirm the
   datastore actually contains it. Documents may be CITED but ABSENT, or a detail may be
   explicitly deferred to a report that is out of scope. These are NOT evidence.

## Classification and anti-fabrication rules
- Sort each retrieved fact into requirement (sending or global), evidence, or gap.
- A requirement with no corresponding satisfied evidence — because it is missing, "in progress",
  "ongoing", scheduled for later (e.g., "before APS", "before drug-substance reception"), or
  deferred to a document not in the datastore — is a GAP. Never invent a passing result.
- Never fabricate values, owners, dates, or evidence. If the datastore does not state something,
  omit it (or use null for gap owner/due_date). Do NOT present a cited-but-absent document or an
  out-of-scope pointer as evidence.
- Preserve every masked/placeholder token EXACTLY as written (e.g., [COMPOUND_1],
  [RECEIVING_SITE_1], [DOC_ID_216], [PERSON_7]). Do not expand, guess, or unmask them.
- Report figures exactly as stated, with units and qualifiers ("NMT", "≤", registered ranges).
  Do not round or normalize.

- Every due_date in a gap must be a milestone phrase that appears verbatim (or near-verbatim)
  in the retrieved source for THAT specific gap. Never reuse a milestone from one gap to fill
  in another. If no milestone is stated for that gap, due_date is null.
- Before adding any regulatory framework, standard, or numeric detail to global_requirements
  or receiving_site_evidence, confirm it was retrieved verbatim from a specific document. If you
  are inferring from general GMP knowledge rather than quoting a retrieved fact, omit it. 

## Sourcing
- Every requirement and evidence entry carries a "source" citing the most specific locator
  available: the document identifier and, if determinable, the section/heading
  (e.g., "[DOC_ID_NEW_3] §9 Dispensing Suite Qualification"). If a fact is synthesized across
  documents (multi-hop), cite all contributing documents in the source string.

## CQAs
- Populate cqa_at_risk only with critical quality attributes the datastore associates with that
  hazard (e.g., product identity; purity & product-related impurities; particulate matter). Use
  the corpus's own wording; do not add CQAs the corpus does not tie to the gap.

## Output contract
- Output EXACTLY ONE JSON object and NOTHING else: no prose, no explanation, no markdown fences.
- The object must have exactly these keys and shapes:

{
  "facility_area": "",
  "sending_site_requirements": [{"requirement": "", "source": ""}],
  "global_requirements": [{"requirement": "", "source": ""}],
  "receiving_site_evidence": [{"evidence": "", "source": ""}],
  "gaps": [{"gap_description": "", "risk_to_product_quality": "", "cqa_at_risk": [], "owner": "", "due_date": ""}]
}

- "facility_area": the evaluated-area label as it appears in the source; use "Facility" if
  unspecified.
- Each list contains one entry per atomic fact; be comprehensive across all Facility sub-topics
  listed above.
- In gaps, "owner" and "due_date" are null unless the datastore states them; "due_date" may be a
  milestone phrase (e.g., "before APS") only if the datastore ties remediation to it.
- "cqa_at_risk" is an array of strings (empty array if none stated).
"""

# ── Registration ───────────────────────────────────────────────────────────────
import google.auth
import vertexai
from vertexai import types
from google.genai import types as genai_types

_, _project_id = google.auth.default()
_LOCATION = "us-central1"
_MODEL_NAME = "gemini-2.5-flash"

_KMS_KEY_NAME = (
    f"projects/{_project_id}/locations/us-central1"
    "/keyRings/poc/cryptoKeys/poc"
)

_client = vertexai.Client(project=_project_id, location=_LOCATION)

_prompt_obj = types.Prompt(
    prompt_data=types.PromptData(
        model=_MODEL_NAME,
        system_instruction=genai_types.Content(
            role="user",
            parts=[genai_types.Part(text=MY_PROMPT_TEXT)],
        ),
        contents=[
            genai_types.Content(
                role="user",
                parts=[genai_types.Part(text="What are the facility gaps for this technology transfer?")],
            )
        ],
    ),
)

_enc_spec = genai_types.EncryptionSpec(kms_key_name=_KMS_KEY_NAME)

# ── Look up existing prompt by display name ────────────────────────────────────
_existing_prompt_id = None
for _ref in _client.prompts.list():
    if getattr(_ref, "display_name", None) == MY_PROMPT_NAME:
        _existing_prompt_id = _ref.prompt_id
        break

if _existing_prompt_id:
    print(f"Found existing prompt '{MY_PROMPT_NAME}' (id: {_existing_prompt_id}) → adding new version")
    _saved = _client.prompts.update(
        prompt_id=_existing_prompt_id,
        prompt=_prompt_obj,
    )
else:
    print(f"No existing prompt named '{MY_PROMPT_NAME}' → creating new prompt resource")
    _create_cfg = types.CreatePromptVersionConfig(
        encryption_spec=_enc_spec,
        prompt_display_name=MY_PROMPT_NAME,
    )
    _saved = _client.prompts.create_version(prompt=_prompt_obj, config=_create_cfg)

_REGISTERED_PROMPT_ID  = _saved.prompt_id
_REGISTERED_VERSION_ID = _saved.version_id

print(f"Registered  : {MY_PROMPT_NAME}")
print(f"Prompt ID   : {_REGISTERED_PROMPT_ID}")
print(f"Version ID  : {_REGISTERED_VERSION_ID}")
print(f"Console     : https://console.cloud.google.com/vertex-ai/generative/language/prompts?project={_project_id}")


Registered  : facilities_generated
Prompt ID   : 7559257065047195648
Version ID  : 2
Console     : https://console.cloud.google.com/vertex-ai/generative/language/prompts?project=dev-mq-tech-transfer


In [11]:
# ── Standalone: verify the registered prompt ──────────────────────────────────
# Run this cell after the registration cell above to confirm the prompt was saved.
# To run in isolation, hard-code _REGISTERED_PROMPT_ID and _REGISTERED_VERSION_ID below.

import google.auth
import vertexai
from google.genai import types as genai_types

_, _project_id = google.auth.default()
_client = vertexai.Client(project=_project_id, location="us-central1")

try:
    _fetched = _client.prompts.get_version(
        prompt_id=_REGISTERED_PROMPT_ID,
        version_id=_REGISTERED_VERSION_ID,
    )
    _sys_instr = _fetched.prompt_data.system_instruction
    if _sys_instr and hasattr(_sys_instr, "parts") and _sys_instr.parts:
        _sys_text = " ".join(p.text for p in _sys_instr.parts if getattr(p, "text", None))
    else:
        _sys_text = str(_sys_instr) if _sys_instr else "(none)"

    print("Verification PASSED")
    print(f"  Prompt ID  : {_fetched.prompt_id}")
    print(f"  Version ID : {_fetched.version_id}")
    print(f"  Model      : {_fetched.prompt_data.model}")
    print(f"  Sys prompt : {_sys_text[:120]}{'...' if len(_sys_text) > 120 else ''}")
except Exception as e:
    print(f"Verification FAILED: {e}")

Verification PASSED
  Prompt ID  : 7559257065047195648
  Version ID : 2
  Model      : gemini-2.5-flash
  Sys prompt : You are a pharmaceutical technology-transfer analyst. Your job is to reconstruct the
FACILITY section of a Technology-Tr...
